# Reading and cleaning chemical data

```{admonition} Learning outcomes
After working through this topic, you should be able to:

1. explain how data are organised in a CSV file
2. read data files with Pandas
3. inspect columns, data types and missing values
4. filter, sort and group chemical data
5. create new columns without overwriting the raw measurements
6. document and justify decisions made during data cleaning
```

Data are everywhere, but data are not automatically knowledge. Before drawing a chemical conclusion, we need to know what the measurements represent, which units were used, and whether the dataset contains missing, duplicated or invalid values.

In this chapter we follow a small UV–Vis experiment. The dataset contains blanks, calibration standards and replicate measurements of an unknown sample. In the next chapters we will use the standards to build a calibration model and estimate the unknown concentration. First we need to read and inspect the raw data.

## Data files

We often store and exchange measurements as plain text because the format is robust and can be read by many programs. Word documents are not plain-text data files because they also contain information about fonts, colours and formatting. `.txt` and `.csv` are common plain-text formats.

```{admonition} Data file
A data file stores observations in a consistent structure so that they can be read and processed by a program. In a tabular data file, each row is usually one observation and the columns describe variables.
```

CSV stands for *comma-separated values*. Each line normally represents one observation, with values separated by commas:

```{code-block} text
sample_id,sample_type,concentration_uM,replicate,absorbance
blank_1,blank,0,1,0.010
blank_2,blank,0,2,0.011
std_2_1,standard,2,1,0.171
std_2_2,standard,2,2,0.169
```

The first row contains column names. The following rows each describe one measurement. This is often called a *tidy* or *long* data layout: one observation per row, one variable per column, and one value per cell.

```{admonition} Units in data
Units should be explicit, but the measurements themselves should normally remain numeric. Here the unit is encoded in the column name `concentration_uM`. An alternative is to keep units in metadata. A value such as `"2 µmol/L"` becomes text and is more difficult to calculate with.
```

## Reading data with Pandas

Pandas is widely used for tabular data. `read_csv` reads a CSV file into a **DataFrame**.

```{admonition} DataFrame
A DataFrame is a two-dimensional data structure with named columns and indexed rows. It resembles a table, but its columns can be selected, calculated with and transformed directly in Python.
```

In [ ]:
import pandas as pd

raw_data = pd.read_csv("data/uvvis_raw.csv")
print(raw_data)

The path `data/uvvis_raw.csv` means that the file is stored in a subfolder named `data` relative to this notebook. If the file is elsewhere, the path needs to be changed.

Some exported files use semicolons as separators and commas as decimal marks. We can describe that explicitly instead of editing the file by hand:

In [ ]:
data = pd.read_csv("data/example_semicolon.csv", sep=";", decimal=",")
print(data)

Explicitly describing the file format makes the analysis more reproducible than manually replacing characters without recording what was changed.

## Inspecting the dataset

Before analysing a file, inspect what Python actually read.

In [ ]:
import pandas as pd

raw_data = pd.read_csv("data/uvvis_raw.csv")

print(raw_data.head())
print(raw_data.tail())
print(raw_data.shape)
print(raw_data.columns)
print(raw_data.dtypes)

- `head()` shows the first five rows.
- `tail()` shows the last five.
- `shape` gives the number of rows and columns.
- `columns` lists the column names.
- `dtypes` shows the data type Pandas assigned to each column.

`info()` combines much of this information:

In [ ]:
raw_data.info()

Text columns are commonly stored as `object` or `string`, integers as `int64`, and decimal numbers as `float64`. A column that should be numeric but appears as `object` may contain text, an unexpected decimal separator or other symbols that cannot be interpreted as numbers.

You can inspect a small UV–Vis dataset in the browser below. The data are included directly in the program so that the example works without uploading a file.

<iframe src="../../basthon/?from=examples/pandas_uvvis_inspect.py" width="100%" height="660" frameborder="0" title="Interactive Python editor for inspecting UV-Vis data with Pandas" loading="lazy" allowfullscreen></iframe>

````{admonition} Check your understanding
:class: tip

1. How many rows and columns are present?
2. Which columns contain text and which contain numbers?
3. Why are the concentration values missing for the unknown sample?
4. Where is the other missing value?

```{admonition} Suggested answer
:class: tip, dropdown
The concentration is missing for the unknown sample because that is the quantity we intend to determine later; this is not necessarily an error. The missing absorbance for one calibration replicate, however, represents a failed or absent measurement and should be investigated before analysis.
```
````

## Preserve the raw data

Raw measurements should normally not be overwritten. If values are changed directly in `raw_data`, it becomes harder to reconstruct what the instrument actually recorded. Make a working copy instead:

In [ ]:
clean_data = raw_data.copy()

A simple workflow is therefore:

1. `raw_data` contains data exactly as read from the file.
2. `clean_data` contains documented processing steps.
3. Analysis is performed on the processed copy.

In a larger project the original raw-data file should also be protected from accidental editing or kept separately. The code then becomes a traceable description of how the raw data were transformed.

## Missing values

Pandas often represents a missing numeric value as `NaN` (*not a number*). Count missing values column by column with:

In [ ]:
print(raw_data.isna().sum())

To inspect the row with a missing absorbance:

In [ ]:
missing_absorbance = raw_data[raw_data["absorbance"].isna()]
print(missing_absorbance)

There is no universally correct automatic response to a missing value. Depending on the experiment, we might:

- inspect the instrument file or laboratory notes
- repeat the measurement
- retain the row and mark the value as missing
- omit the row from one particular analysis
- estimate the value only when there is a defensible scientific and statistical reason

In this teaching dataset, the instrument log states that measurement `std_6_3` failed because the cuvette was incorrectly positioned. We therefore omit that row from analyses that require an absorbance value:

In [ ]:
clean_data = raw_data.copy()
clean_data = clean_data.dropna(subset=["absorbance"])

Using `subset` is important. `dropna()` without arguments would also remove the unknown samples because their concentrations are intentionally missing.

```{admonition} Important
A missing value is not the same as a measured value of zero. Zero may be a valid experimental result. `NaN` means the value is unavailable.
```

## Duplicates

A row may accidentally have been stored twice. We can check for exact duplicates with:

In [ ]:
duplicates = raw_data.duplicated()
print(raw_data[duplicates])

An identical row is not automatically an error. Two genuine replicate measurements can sometimes have exactly the same numerical result. Removing a duplicate should therefore be based on how the data were generated, not only on identical values.

If an accidental duplicated export row has been documented, we can remove it with `drop_duplicates()`.

In [ ]:
clean_data = clean_data.drop_duplicates()

## Filtering data

Logical conditions select rows. Here we extract only the calibration standards:

In [ ]:
standards = clean_data[clean_data["sample_type"] == "standard"]
print(standards)

Several conditions can be combined. Parentheses around each condition are important when using Pandas with `&` and `|`.

In [ ]:
mid_range = clean_data[
    (clean_data["sample_type"] == "standard")
    & (clean_data["concentration_uM"] >= 4)
    & (clean_data["concentration_uM"] <= 8)
]
print(mid_range)

Filtering is not the same as deleting. We are creating a view or a new DataFrame for a particular scientific question while the full raw dataset remains available.

## Sorting data

Rows can be sorted by one or more columns:

In [ ]:
sorted_standards = standards.sort_values(["concentration_uM", "replicate"])
print(sorted_standards)

Sorting does not change the chemical meaning of the observations, but it can make unexpected values easier to notice and makes it easier to verify that replicates are associated with the correct concentration.

## Grouping and summarising data

The calibration standards were measured in replicates. We can group by concentration and calculate the number of measurements, mean and sample standard deviation.

In [ ]:
summary = (
    standards
    .groupby("concentration_uM")["absorbance"]
    .agg(["count", "mean", "std"])
)
print(summary)

Grouping is powerful because one operation is applied consistently to every concentration. But a summary table is not a replacement for the raw measurements: a mean and standard deviation can hide unusual replicates or missing data.

You can clean and group a small embedded dataset in the editor below.

<iframe src="../../basthon/?from=examples/pandas_uvvis_clean.py" width="100%" height="700" frameborder="0" title="Interactive Python editor for cleaning and grouping UV-Vis data" loading="lazy" allowfullscreen></iframe>

## Creating new columns

Some processing steps produce new quantities that should be stored without erasing the original measurements. For example, we can correct absorbance for the mean blank signal.

In [ ]:
blank_mean = clean_data.loc[clean_data["sample_type"] == "blank", "absorbance"].mean()

clean_data["blank_corrected_absorbance"] = clean_data["absorbance"] - blank_mean
print(clean_data.head())

Keeping both columns makes the transformation explicit: `absorbance` is the recorded value, whereas `blank_corrected_absorbance` is derived.

The same principle applies to unit conversions. If a dataset stores concentration in µmol/L but a later calculation needs mol/L, create a new column rather than silently replacing the original.

In [ ]:
clean_data["concentration_M"] = clean_data["concentration_uM"] * 1e-6

### Cleaning text categories

Real datasets often contain inconsistent capitalisation or whitespace, such as `Standard`, `standard ` and `STANDARD`. These can be normalised if they truly refer to the same category.

In [ ]:
clean_data["sample_type"] = clean_data["sample_type"].str.strip().str.lower()

Always inspect the result after an automatic text-cleaning operation. Two labels that look similar may represent genuinely different sample types.

## From DataFrame to figure

Pandas columns can be passed directly to Matplotlib. Here we show every calibration replicate so that the variation remains visible.

In [ ]:
import matplotlib.pyplot as plt

standards = clean_data[clean_data["sample_type"].isin(["blank", "standard"])]

plt.scatter(standards["concentration_uM"], standards["absorbance"])
plt.xlabel("Concentration (µmol/L)")
plt.ylabel("Absorbance")
plt.tight_layout()
plt.show()

If we plotted only the group means, the figure would be tidier, but we would lose direct information about replicate-to-replicate variation. Whether to show raw observations, summaries or both depends on the purpose of the figure.

## A reproducible workflow

The complete processing sequence can be written as ordinary code rather than manual spreadsheet edits.

In [ ]:
import pandas as pd

# 1. Read and preserve raw data
raw_data = pd.read_csv("data/uvvis_raw.csv")

# 2. Inspect
print(raw_data.info())
print(raw_data.isna().sum())

# 3. Make a working copy
clean_data = raw_data.copy()

# 4. Apply a documented decision: remove failed absorbance measurement
clean_data = clean_data.dropna(subset=["absorbance"])

# 5. Add derived values rather than overwriting raw measurements
blank_mean = clean_data.loc[clean_data["sample_type"] == "blank", "absorbance"].mean()
clean_data["blank_corrected_absorbance"] = clean_data["absorbance"] - blank_mean

# 6. Save processed data without overwriting the raw file
clean_data.to_csv("data/uvvis_processed.csv", index=False)

The code documents **what** was done. A complete scientific analysis must also explain **why** each decision was justified by the experimental context.

```{admonition} A useful principle
Data cleaning should make the dataset more faithful to the experiment, not merely make the later statistical result look nicer.
```

## Exercises

```{admonition} Exercise 2.1
:class: tip
Open `data/uvvis_raw.csv` as plain text. Identify the column names, delimiter, missing values and the columns in which units are encoded.
```

```{admonition} Exercise 2.2
:class: tip
Read `data/uvvis_raw.csv` with Pandas. Use `head`, `shape`, `columns`, `dtypes` and `info` to write a short description of the dataset.
```

```{admonition} Exercise 2.3
:class: tip
Count missing values. Explain why the missing concentrations for the unknown sample and the missing absorbance for `std_6_3` have different scientific meanings.
```

```{admonition} Exercise 2.4
:class: tip
Create `clean_data` as a copy of the raw data. Remove only rows that lack absorbance. Verify that the unknown samples are still present.
```

```{admonition} Exercise 2.5
:class: tip
Filter the calibration standards and group them by concentration. Calculate `count`, `mean` and `std` for absorbance. Which concentration has fewer valid replicates than the others, and why?
```

```{admonition} Exercise 2.6
:class: tip
Calculate the mean blank absorbance and add a new column containing blank-corrected absorbance. Explain why it is preferable to keep the uncorrected measurements as a separate column.
```

```{admonition} Exercise 2.7
:class: tip
The file `data/reaction_kinetics.csv` contains time and absorbance. Read it, inspect the data types, sort by time and add a column `relative_absorbance` in which each absorbance is divided by the initial value. Plot the result.
```

```{admonition} Exercise 2.8 — cleaning decisions
:class: tip
Imagine that one calibration replicate is much higher than the other two but is not missing. List at least four things you should investigate before deciding whether to exclude it. Which of those questions require chemical or experimental information that Pandas cannot supply?
```